# Kapitel 6 — Ein Modell bewerten und die Unsicherheit messen · Kommentierte Lösung

*Mathe für KI, Schritt für Schritt*, Belk Siri — Begleit-Notebook.

[In Google Colab öffnen](https://colab.research.google.com/github/belksiri/books/blob/main/maths-pour-l-ia-pas-a-pas/notebooks/de/ch06_corrige.ipynb) · Begleitseite: https://belksiri.github.io/books/maths-pour-l-ia-pas-a-pas/de.html

Jede Antwort beginnt mit der Überlegung auf Papier und zeigt dann ihre Entsprechung in Python. Die Kommentare geben bei jeder Rechnung an, welcher Datensatz verwendet wird.

**Inhalt.** Daten · Die Rechnungen der Lektionen 6.1 bis 6.6 · Aufgaben 6.1 bis 6.6

**Das Protokoll, im ganzen Notebook:** auf den **Trainingsdaten** anpassen, auf den **Validierungsdaten** auswählen, **ein einziges Mal** auf den **Testdaten** bewerten. Für einen A/B-Test: zwei **zufällig zugeteilte** Gruppen vergleichen und die Differenz **mit ihrem Konfidenzintervall** angeben.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Farben des Buchs: eine Farbe pro Datensatz
SET_TRAIN, SET_VAL, SET_TEST = '#2e7d32', '#b84d00', '#1565c0'   # Training (grün), Validierung (dunkelorange), Test (blau)
SPAM, LEGIT = '#d62728', '#b9b6ae'
VERSION_A, VERSION_B, RESIDUAL, THRESHOLD = '#757575', '#00897b', '#c2185b', '#8e24aa'   # A/B-Test: A grau, B petrol


def sigmoid(z):
    return 1 / (1 + np.exp(-np.asarray(z, dtype=float)))


def mean_loss(labels, X, w, b):
    """Mittlerer logarithmischer Verlust (Gleichungen 5.6 und 5.7)."""
    p = np.clip(sigmoid(X @ np.asarray(w, dtype=float) + b), 1e-12, 1 - 1e-12)
    y = np.asarray(labels, dtype=float)
    return float(np.mean(-(y * np.log(p) + (1 - y) * np.log(1 - p))))


FEATURE_NAMES = ['links', 'suspicious_words', 'length', 'capitals', 'hour', 'attachments', 'size', 'recipients']


def make_messages(seed=77):
    """Fiktive Nachrichten (Aufgabe 6.4 und Lektion 6.4): 40 Gruppen, Spam-Kampagnen (gerade Gruppen) oder
    legitime Absender (ungerade Gruppen). Die Nachrichten einer Gruppe sind fast identisch. Nur die ersten
    beiden Merkmale hängen vom Label ab; die acht Merkmale haben sehr unterschiedliche Einheiten."""
    rng = np.random.default_rng(seed)
    units = np.array([2, 1.5, 30, 0.2, 5, 1, 50, 3])
    offsets = np.array([3, 2, 200, 1, 12, 0, 500, 10])
    rows = []
    for g in range(40):
        spam = g % 2 == 0
        size = rng.integers(4, 15)
        base = rng.normal(0, 1, 8)
        base[:2] += 1.0 if spam else -1.0
        for _ in range(size):
            rows.append([g, int(spam)] + list((base + rng.normal(0, 0.3, 8)) * units + offsets))
    return pd.DataFrame(rows, columns=['group', 'spam'] + FEATURE_NAMES)


def fit(X, y, lam, l1=False, step_k=0.1, n_steps=3000):
    """Gradientenabstieg auf dem bestraften Verlust: L2 (Gleichung 6.2) oder L1 (Gleichung 6.3). b wird nicht bestraft."""
    w, b = np.zeros(X.shape[1]), 0.0
    for _ in range(n_steps):
        gap = sigmoid(X @ w + b) - y
        gw, gb = X.T @ gap / len(y), gap.mean()
        if l1:   # Gradientenschritt, dann jedes Gewicht um höchstens pas × λ an 0 heranrücken (ohne Vorzeichenwechsel)
            w = w - step_k * gw
            w = np.sign(w) * np.maximum(np.abs(w) - step_k * lam, 0)
        else:    # die Ableitung von λ × Summe der w² ist 2λw
            w = w - step_k * (gw + 2 * lam * w)
        b -= step_k * gb
    return w, b

## Daten

Die Bezeichnungen D6.1 bis D6.5, D6.E1, D6.E2, D6.E3, D6.E5 und D5.1 sind die des Buchs und der CSV-Dateien.

In [ ]:
# Daten von Kapitel 6, übernommen aus den CSV-Dateien des Buchs (nicht verändern)

# D6.1 — die drei Datensätze des Vereins: Nachrichten, Spams und Fehler des Filters für einen Schwellenwert
subsets = pd.DataFrame([
    ('entrainement', 600, 120, 0.7, 30, 3),
    ('validation', 200, 40, 0.3, 4, 14),
    ('validation', 200, 40, 0.5, 8, 6),
    ('validation', 200, 40, 0.7, 15, 2),
    ('test', 250, 50, 0.7, 20, 4)
], columns=['subset', 'messages', 'spams', 'threshold', 'fn', 'fp'])

# D6.2 — zwei Aufteilungen derselben Nachrichten: die Testdaten, beim Schwellenwert 0,7
leak = pd.DataFrame([
    ('aleatoire', 250, 50, 25, 1, 10, 4),
    ('par_campagne', 250, 50, 0, 0, 20, 4)
], columns=['split', 'messages', 'spams', 'copies_seen', 'missed_copies', 'missed_new', 'fp'])

# D6.3 — sechs neue Nachrichten (Validierung in Lektion 6.3)
six = pd.DataFrame([
    ('R1', 'spam', 4, 2),
    ('R2', 'legitime', 1, 3),
    ('R3', 'spam', 3, 4),
    ('R4', 'legitime', 0, 0),
    ('R5', 'spam', 5, 3),
    ('R6', 'legitime', 2, 1)
], columns=['message', 'label', 'links', 'suspicious_words'])

# D5.1 — die zwölf Nachrichten aus Kapitel 5 (Training in Lektion 6.3)
twelve = pd.DataFrame([
    ('M1', 'spam', 5, 4),
    ('M2', 'legitime', 2, 0),
    ('M3', 'legitime', 4, 0),
    ('M4', 'spam', 3, 3),
    ('M5', 'legitime', 1, 0),
    ('M6', 'legitime', 2, 2),
    ('M7', 'spam', 2, 3),
    ('M8', 'legitime', 2, 1),
    ('M9', 'spam', 4, 4),
    ('M10', 'legitime', 1, 1),
    ('M11', 'legitime', 0, 0),
    ('M12', 'legitime', 0, 1)
], columns=['message', 'label', 'links', 'suspicious_words'])

# D6.E2 — die Mediathek (Aufgabe 6.2): Validierungs- und Testdaten
media_library = pd.DataFrame([
    ('validation', 150, 30, 0.4, 3, 18),
    ('validation', 150, 30, 0.5, 6, 9),
    ('validation', 150, 30, 0.6, 10, 5),
    ('validation', 150, 30, 0.7, 15, 3),
    ('test', 240, 48, 0.4, 6, 26),
    ('test', 240, 48, 0.5, 10, 14),
    ('test', 240, 48, 0.6, 17, 9),
    ('test', 240, 48, 0.7, 22, 3)
], columns=['subset', 'messages', 'spams', 'threshold', 'fn', 'fp'])

# D6.4 — die drei Folds des Vereins (Lektion 6.4): Links (Mittelwert, Standardabweichung) und Validierungsverluste
folds = pd.DataFrame([
    (1, 270, 54, 2.10, 1.60, 0.29, 0.30),
    (2, 264, 52, 2.40, 1.90, 0.38, 0.33),
    (3, 266, 54, 2.20, 1.70, 0.33, 0.31)
], columns=['fold', 'messages', 'spams', 'mean_links', 'std_links', 'loss_l001', 'loss_l01'])

# D6.5 — der A/B-Test der Buchhandlung (Lektionen 6.5 und 6.6)
bookshop = pd.DataFrame([
    ('A', 'aktueller Brief', 2500, 200),
    ('B', 'kostenloser Versand', 2400, 240)
], columns=['version', 'description', 'customers', 'orders'])

# D6.E1 — Kreuzvalidierung der Mediathek (Aufgabe 6.1): Verlust jedes Folds für fünf λ
folds_lambda = pd.DataFrame([
    (1, 0.001, 0.52),
    (1, 0.01, 0.44),
    (1, 0.1, 0.36),
    (1, 1, 0.35),
    (1, 10, 0.52),
    (2, 0.001, 0.40),
    (2, 0.01, 0.37),
    (2, 0.1, 0.34),
    (2, 1, 0.38),
    (2, 10, 0.55),
    (3, 0.001, 0.61),
    (3, 0.01, 0.49),
    (3, 0.1, 0.39),
    (3, 1, 0.37),
    (3, 10, 0.51)
], columns=['fold', 'lambda', 'loss'])

# D6.E3 — Verteilung von vier Gruppen auf die Folds des Ehrenamtlichen (Aufgabe 6.3)
folds_of_groups = pd.DataFrame([
    ('Montres', 'Spam-Kampagne', 9, 8, 9),
    ('Colis en attente', 'Spam-Kampagne', 5, 6, 4),
    ('Loterie', 'Spam-Kampagne', 7, 7, 8),
    ('Lettre de la mairie', 'legitimer Absender', 4, 3, 5)
], columns=['group', 'kind', 'fold_1', 'fold_2', 'fold_3'])

# D6.E5 — der A/B-Test des Theaters (Aufgabe 6.5)
theater = pd.DataFrame([
    ('A', 'üblicher Brief', 1800, 126),
    ('B', 'Brief mit Video', 1750, 147)
], columns=['version', 'description', 'audience', 'bookings'])

### Die Variablen der Daten

Die Namen im Code sind englisch. Für die Spalten gibt die letzte Spalte den Namen an, der in den CSV-Dateien des Buchs verwendet wird.

| Name | Bedeutung | Einheit | Spalte der CSV-Datei |
|:--|:--|:--|:--|
| `message` | Kennung der Nachricht (M1…, R1…) | — | — |
| `label` | tatsächliches Label: Spam oder legitim | — | `etiquette` |
| `links` | Anzahl der Links in der Nachricht | Links | `liens` |
| `suspicious_words` | Anzahl der verdächtigen Wörter in der Nachricht | Wörter | `mots_suspects` |
| `messages` | Anzahl der Nachrichten | Nachrichten | — |
| `spams` | Anzahl der Spams | Nachrichten | — |
| `subset` | Datensatz: Training, Validierung oder Test | — | `lot` |
| `threshold` | Entscheidungsschwellenwert | — | `seuil` |
| `campaign` | Versandkampagne der Nachricht | — | `campagne` |
| `group` | Gruppe von Nachrichten (eine ganze Kampagne oder ein Absender), die vollständig in einem Fold bleibt | — | `groupe` |
| `kind` | Art der Gruppe: Spam-Kampagne oder legitimer Absender | — | `nature` |
| `fold` | Nummer des Folds der Kreuzvalidierung | — | `pli` |
| `fold_1` | D6.E3: Anzahl der Nachrichten der Gruppe in Fold 1 | Nachrichten | `pli_1` |
| `fold_2` | D6.E3: Anzahl der Nachrichten der Gruppe in Fold 2 | Nachrichten | `pli_2` |
| `fold_3` | D6.E3: Anzahl der Nachrichten der Gruppe in Fold 3 | Nachrichten | `pli_3` |
| `mean_links` | D6.4: Mittelwert der Links über die Trainingsnachrichten der Runde (die beiden anderen Folds) | Links | `moyenne_liens` |
| `std_links` | D6.4: Standardabweichung der Links über die Trainingsnachrichten der Runde (die beiden anderen Folds) | Links | `ecart_type_liens` |
| `description` | Beschreibung der getesteten Version | — | — |
| `audience` | Anzahl der Zuschauer | Personen | `spectateurs` |
| `bookings` | Anzahl der Reservierungen | Reservierungen | `reservations` |
| `customers` | Anzahl der Kunden, denen die Version gezeigt wurde | Kunden | `clients` |
| `orders` | Anzahl der Bestellungen | Bestellungen | `commandes` |
| `version` | Version der Kampagne (A oder B) | — | — |
| `twelve` | die zwölf Nachrichten aus Kapitel 5 | — | — |
| `loss_l001` | D6.4: Validierungsverlust des Folds, mit λ = 0,01 | — | `perte_l001` |
| `loss_l01` | D6.4: Validierungsverlust des Folds, mit λ = 0,1 | — | `perte_l01` |
| `fn` | Anzahl der falsch Negativen (verpasste Spams) | Nachrichten | — |
| `fp` | Anzahl der falsch Positiven (markierte legitime Nachrichten) | Nachrichten | — |
| `split` | D6.2: Aufteilung des Testdatensatzes (zufällig oder nach Kampagne) | — | `decoupage` |
| `copies_seen` | D6.2: Test-Spams, kopiert aus einer im Training gesehenen Kampagne | Nachrichten | `copies_vues` |
| `missed_copies` | D6.2: vom Filter verpasste Kopien | Nachrichten | `manques_copies` |
| `missed_new` | D6.2: vom Filter verpasste neue Spams | Nachrichten | `manques_nouveaux` |
| `lambda` | D6.E1: Stärke λ der Strafe | — | — |
| `loss` | D6.E1: Validierungsverlust des Folds für dieses λ | — | `perte` |

## Die Rechnungen der Lektionen, in Python

Diese Zellen wiederholen die Rechnungen der Lektionen 6.1 bis 6.6.

In [ ]:
# Lektion 6.1 — Kosten pro Nachricht (Gleichung 6.1), k = 3 für den Verein
k = 3
t = subsets.copy()
t['Kosten'] = t['fn'] + k * t['fp']
t['Kosten pro Nachricht'] = (t['Kosten'] / t['messages']).round(3)    # geteilt durch die Größe DES DATENSATZES
print(t.to_string(index=False))
val = t[t['subset'] == 'validation']
chosen = val.loc[val['Kosten pro Nachricht'].idxmin(), 'threshold']
print('auf den Validierungsdaten gewählter Schwellenwert:', chosen)
test = t[(t['subset'] == 'test') & (t['threshold'] == chosen)].iloc[0]
print(f"Test, ein einziges Mal: Kosten {test['Kosten']} bei {test['messages']} Nachrichten = {test['Kosten pro Nachricht']:.3f} pro Nachricht")

a7 = t[t['threshold'] == 0.7]
colors = {'entrainement': SET_TRAIN, 'validation': SET_VAL, 'test': SET_TEST}
plt.figure(figsize=(5, 2.8))
plt.bar(a7['subset'], a7['Kosten pro Nachricht'], color=[colors[l] for l in a7['subset']])
plt.ylabel('Kosten pro Nachricht (Schwellenwert 0,7)'); plt.show()

In [ ]:
# Lektion 6.2 — ein Datenleck: Recall und Kosten pro Nachricht der beiden Aufteilungen
for _, l in leak.iterrows():
    missed = l['missed_copies'] + l['missed_new']
    new_spams = l['spams'] - l['copies_seen']
    print(f"{l['split']:>12} : Recall = {(l['spams'] - missed) / l['spams']:.2f}, "
          f"Kosten pro Nachricht = {(missed + 3 * l['fp']) / l['messages']:.3f}, "
          f"Anteil verpasster neuer Spams = {l['missed_new']} / {new_spams} = {l['missed_new'] / new_spams:.2f}")

# Die Kampagnen aufteilen, nicht die Nachrichten: kleine Demonstration mit 40 Kampagnen unterschiedlicher Größe
rng = np.random.default_rng(6)
campaign = np.repeat(np.arange(40), rng.integers(5, 40, size=40))          # Kampagnennummer jedes Spams
at_random = rng.random(campaign.size) < 0.25                              # 25 % der NACHRICHTEN in den Test
test_campaigns = rng.choice(40, size=10, replace=False)                         # 25 % der KAMPAGNEN in den Test
by_campaign = np.isin(campaign, test_campaigns)
for name, test_ in (('zufällig', at_random), ('je Kampagne', by_campaign)):
    shared = len(set(campaign[test_]) & set(campaign[~test_]))
    print(f'{name:>12} : {test_.sum()} Spams im Test, {shared} Kampagnen sowohl im Test als auch anderswo vorhanden')

In [ ]:
# Lektion 6.3 — Überanpassung: Filter A und B auf den Trainingsdaten (zwölf Nachrichten) und den Validierungsdaten (sechs)
X12, y12 = twelve[['links', 'suspicious_words']].to_numpy(float), (twelve['label'] == 'spam').to_numpy(int)
X6, y6 = six[['links', 'suspicious_words']].to_numpy(float), (six['label'] == 'spam').to_numpy(int)
filters = {'A': ((1, 0.5), -4), 'B': ((-0.54, 4.39), -9.17)}
for name, (w, b) in filters.items():
    p6 = sigmoid(X6 @ np.array(w) + b)
    losses6 = -(y6 * np.log(p6) + (1 - y6) * np.log(1 - p6))
    print(f'Filter {name}: Training {mean_loss(y12, X12, w, b):.2f}, Validierung {mean_loss(y6, X6, w, b):.2f} '
          f'| Verluste der sechs: {np.round(losses6, 2)}')

# Bestrafter L2-Verlust (Gleichung 6.2), λ = 0,05; die Konstante b wird nicht bestraft
lam = 0.05
for name, (w, b) in filters.items():
    penalty = lam * np.sum(np.square(w))
    print(f'Filter {name}: Summe der quadrierten Gewichte = {np.sum(np.square(w)):.4f}, Strafterm = {penalty:.4f}, '
          f'bestrafter Verlust = {mean_loss(y12, X12, w, b) + penalty:.2f}')

# Der Strafterm hängt von den Einheiten ab: Links in Zehnern gezählt, gleiche Vorhersagen, anderer Strafterm
X12_tens = X12 / np.array([10, 1])
w_tens = (10, 0.5)
print('gleiche Wahrscheinlichkeiten:', np.allclose(sigmoid(X12_tens @ np.array(w_tens) - 4), sigmoid(X12 @ np.array([1, 0.5]) - 4)),
      f'| Strafterm in Einern: {lam * (1 + 0.25):.4f}; in Zehnern: {lam * (100 + 0.25):.4f}')

In [ ]:
# Lektion 6.4 — Standardisierung in jeder Runde: Mittelwert und Standardabweichung der Links auf den BEIDEN ANDEREN Folds
for k in range(3):
    others = folds[folds['fold'] != k + 1]
    n = others['messages'].sum()
    m = (others['messages'] * others['mean_links']).sum() / n
    # Standardabweichung der Vereinigung zweier Folds (Division durch n): Mittelwert der Quadrate minus Quadrat des Mittelwerts
    v = (others['messages'] * (others['std_links'] ** 2 + others['mean_links'] ** 2)).sum() / n - m ** 2
    print(f'Runde {k + 1}: Validierung = Fold {k + 1}, Training = {n} Nachrichten, Mittelwert {m:.2f}, Standardabweichung {np.sqrt(v):.2f}')
print(f'Nachricht mit 6 Links aus Fold 3, in Runde 3: (6 - 2,25) / 1,76 = {(6 - 2.25) / 1.76:.2f}')

# Kreuzvalidierungswert (Gleichung 6.4) der beiden Werte von λ
for col, lam in (('loss_l001', 0.01), ('loss_l01', 0.1)):
    print(f'λ = {lam}: Verluste der Folds {list(folds[col])}, KV = {folds[col].mean():.3f}')

In [ ]:
# Lektion 6.4 (Schluss) — L1 und L2 auf fiktiven Nachrichten mit acht Merkmalen (dieselben wie in Aufgabe 6.4)
msgs = make_messages()
X8 = msgs[FEATURE_NAMES].to_numpy(float)
X8 = (X8 - X8.mean(0)) / X8.std(0)        # hier Standardisierung über alle Nachrichten: nur zur Demonstration
y8 = msgs['spam'].to_numpy(float)
w_l2, _ = fit(X8, y8, 0.1)
w_l1, _ = fit(X8, y8, 0.1, l1=True)
print('Gewichte L2:', np.round(w_l2, 2))
print('Gewichte L1:', np.round(w_l1, 2))
print(f'Nullgewichte: L2 {int(np.sum(np.abs(w_l2) < 1e-9))}, L1 {int(np.sum(np.abs(w_l1) < 1e-9))} '
      '(nur die ersten beiden Merkmale hängen mit dem Label zusammen)')

In [ ]:
# Lektionen 6.5 und 6.6 — der A/B-Test der Buchhandlung
a, b_ = bookshop.iloc[0], bookshop.iloc[1]
pA, pB = a['orders'] / a['customers'], b_['orders'] / b_['customers']
gap = pB - pA
print(f'Rate A = {100 * pA:.1f} %, Rate B = {100 * pB:.1f} %, Differenz = {100 * gap:.1f} Punkte, '
      f'relative Veränderung = {100 * gap / pA:.0f} %')
seA, seB = np.sqrt(pA * (1 - pA) / a['customers']), np.sqrt(pB * (1 - pB) / b_['customers'])
se = np.sqrt(seA ** 2 + seB ** 2)                          # Gleichung 6.6: man addiert die QUADRATE
low, high = gap - 1.96 * se, gap + 1.96 * se           # Gleichung 6.7
print(f'SF von A = {100 * seA:.2f} Punkt, SF von B = {100 * seB:.2f} Punkt, SF der Differenz = {100 * se:.2f} Punkt')
print(f'95 %-Konfidenzintervall: von {100 * low:.1f} bis {100 * high:.1f} Punkten')
print(f'für 40 000 Abonnenten: {40000 * gap:.0f} Bestellungen mehr (von {40000 * low:.0f} bis {40000 * high:.0f})')

plt.figure(figsize=(6, 1.8))
plt.axvline(0, color='black', linewidth=2.5, label='0: kein Effekt')
plt.axvline(1, color=THRESHOLD, linestyle='--', label='Rentabilitätsschwelle: 1 Punkt')
plt.plot([100 * low, 100 * high], [0, 0], color=RESIDUAL, linewidth=4, label='95 %-Intervall')
plt.scatter([100 * gap], [0], marker='D', s=70, color=RESIDUAL, zorder=3)
plt.yticks([]); plt.xlim(-1, 5); plt.xlabel('Effekt der Version B (Prozentpunkte)')
plt.legend(fontsize=7, loc='upper right'); plt.show()

# Lektion 6.6 — die Kosten pro Nachricht im Test (Lektion 6.1) sind ein Mittelwert: gleiche Überlegung
costs = np.array([1] * 20 + [3] * 4 + [0] * 226)            # 20 verpasste Spams, 4 Fehlalarme, 226 ohne Kosten
se_c = costs.std() / np.sqrt(len(costs))                     # Standardabweichung (Division durch n) / Wurzel aus n
print(f'Kosten pro Nachricht = {costs.mean():.3f}, Standardabweichung = {costs.std():.3f}, SF = {se_c:.3f}, '
      f'Intervall: von {costs.mean() - 1.96 * se_c:.2f} bis {costs.mean() + 1.96 * se_c:.2f}')

## Aufgabe 6.1 — Eine Grafik lesen: drei Folds, fünf Werte von λ

*Begriffe: Kreuzvalidierung (Lektion 6.4, Gleichung 6.4), Überanpassung und Strafterm (Lektion 6.3).* Daten D6.E1: der Validierungsverlust jedes Folds der Mediathek, für fünf Werte von $\lambda$.

### a) bis e) Kommentierte Lösung

**a)** Fold 2 für $\lambda = 0{,}1$: 0,34. Fold 3 für $\lambda = 0{,}001$: 0,61.

**b)** Die Mittelwerte der drei Folds betragen etwa 0,51, 0,43, 0,36, 0,37 und 0,53. Der kleinste ist der von $\lambda = 0{,}1$ (0,363). Für sich genommen hätte Fold 1 $\lambda = 1$ gewählt (0,35), Fold 2 $\lambda = 0{,}1$ (0,34) und Fold 3 $\lambda = 1$ (0,37): Zwei von drei Folds bevorzugen einen anderen Wert als der Mittelwert.

**c)** Bei $\lambda = 0{,}001$ bremst der Strafterm fast nichts: Der Filter lernt die Besonderheiten der Trainings-Folds auswendig, wie Filter B in Lektion 6.3, und irrt sich auf dem Validierungs-Fold mit großer Sicherheit. Bei $\lambda = 10$ drückt der Strafterm alle Gewichte gegen 0: Der Filter gibt fast allen Nachrichten dieselbe Wahrscheinlichkeit und unterscheidet die Spams nicht mehr. Dazwischen durchläuft der Verlust ein Tal.

**d)** Die Mittelwerte von $\lambda = 0{,}1$ und $\lambda = 1$ unterscheiden sich um etwa 0,003. Für dasselbe $\lambda$ unterscheiden sich die Folds viel stärker: von 0,34 bis 0,39 für $\lambda = 0{,}1$. Die Wahl zwischen diesen beiden Werten ist also nicht belastbar: Nach den Daten sind sie gleich gut. Dagegen ist es eine belastbare Entscheidung, 0,001 und 10 auszuschließen.

**e)** Die 0,36 ist der Kreuzvalidierungswert, mit dem $\lambda$ **gewählt** wurde: Er schmeichelt der gewählten Einstellung leicht. Die Mediathek muss zuerst das Endmodell mit dem gewählten $\lambda$ auf allen Folds anpassen, ihren Schwellenwert festlegen und dann den Verlust **ein einziges Mal** auf zurückgehaltenen Testdaten messen. Diese Zahl darf sie angeben.

In [ ]:
means = folds_lambda.groupby('lambda')['loss'].mean()     # KV(λ): Mittelwert der drei Folds
lam_chosen = means.idxmin()

In [ ]:
# Überprüfung von a) und b)
if means is None or lam_chosen is None:
    print('noch nicht ausgefüllt. Ergänzen Sie die beiden Zeilen.')
else:
    print('Mittelwerte:', means.round(3).to_dict(), '→',
          'richtig' if np.allclose(means.values, [0.51, 1.3 / 3, 1.09 / 3, 1.1 / 3, 1.58 / 3]) else 'bitte prüfen')
    print('gewähltes λ:', lam_chosen, '→', 'richtig' if lam_chosen == 0.1 else 'bitte prüfen')
    print('Wahl jedes einzelnen Folds:', folds_lambda.loc[folds_lambda.groupby('fold')['loss'].idxmin(), ['fold', 'lambda']].values.tolist())
    for k, (style, marker) in enumerate((('-', 'o'), ('--', 's'), (':', '^'))):
        d = folds_lambda[folds_lambda['fold'] == k + 1]
        plt.plot(d['lambda'], d['loss'], style, marker=marker, color=SET_VAL, label=f'Fold {k + 1}')
    plt.plot(means.index, means.values, 'D-', color='black', linewidth=2.5, label='Mittelwert')
    plt.xscale('log'); plt.xlabel('λ'); plt.ylabel('Validierungsverlust'); plt.legend(); plt.grid(alpha=0.4); plt.show()

## Aufgabe 6.2 — Einen Schwellenwert wählen und dann bewerten: die Mediathek

*Begriffe: drei Datensätze (Lektion 6.1), Kosten pro Nachricht (Gleichung 6.1), Kosten eines Schwellenwerts (Gleichung 5.9).* Für die Mediathek kostet ein Fehlalarm so viel wie zwei verpasste Spams: $k = 2$. Daten D6.E2. Bearbeiten Sie zuerst die Fragen a) bis e) auf Papier, **ohne vor Frage b) die Zeilen der Testdaten anzusehen**.

### a) bis e) Kommentierte Lösung

**a)** Auf den **Validierungsdaten** (150 Nachrichten), mit $FN + 2 \times FP$:

| Schwellenwert | Kosten | Kosten pro Nachricht |
|:--|--:|--:|
| 0,4 | $3 + 2 \times 18 = 39$ | $39 / 150 = 0{,}260$ |
| 0,5 | $6 + 2 \times 9 = 24$ | $24 / 150 = 0{,}160$ |
| 0,6 | $10 + 2 \times 5 = 20$ | $20 / 150 \approx 0{,}133$ |
| 0,7 | $15 + 2 \times 3 = 21$ | $21 / 150 = 0{,}140$ |

Der Schwellenwert **0,6** kostet auf den Validierungsdaten am wenigsten. Wir behalten ihn.

**b)** Auf den **Testdaten** (240 Nachrichten), nur beim Schwellenwert 0,6: $17 + 2 \times 9 = 35$, dann $35 / 240 \approx 0{,}146$ pro Nachricht. Das ist die Zahl, die angegeben wird.

**c)** Die beiden Datensätze sind nicht gleich groß: 150 und 240 Nachrichten. Die Summen, 20 und 35, würden glauben lassen, dass der Filter im Test 75 % mehr kostet. Pro Nachricht liegt der Unterschied nur zwischen 0,133 und 0,146, also etwa 9 % mehr. Eine Summe wächst mit der Zahl der Nachrichten; Kosten pro Nachricht nicht.

**d)** Ja. Der Schwellenwert wurde gewählt, weil er auf genau diesen 150 Nachrichten am besten abschnitt, mit ihrem Anteil an Glück. Seine Leistung auf den Validierungsdaten ist daher leicht geschönt. Die Testdaten, die für keine Entscheidung verwendet wurden, profitieren nicht von diesem Glück.

**e)** Erster Fehler: Er **wählt den Schwellenwert auf den Testdaten**, die damit zu einer zweiten Validierung werden. Zweiter Fehler: Er gibt die **besten** der vier Testkosten an, 0,117, und die sind zwangsläufig geschönt: Das ist ein Minimum, keine Messung. Er sollte 0,146 pro Nachricht angeben, beim auf den Validierungsdaten gewählten Schwellenwert 0,6. Wenn er 0,6 und 0,7 unbedingt vergleichen will, braucht er größere Validierungsdaten oder eine Kreuzvalidierung und danach neue Testdaten. Beachten Sie übrigens, dass sich 0,6 und 0,7 auf den Validierungsdaten nur um eine Kosteneinheit bei 150 Nachrichten unterscheiden: Die Wahl zwischen ihnen ist wackelig.

In [ ]:
def cost_per_message(fn, fp, n, k):
    return (fn + k * fp) / n                    # geteilt durch die Größe des Datensatzes


def threshold_choice(validation_table, k):
    """Gibt den Schwellenwert mit den geringsten Kosten pro Nachricht auf den Validierungsdaten zurück."""
    costs = [cost_per_message(l.fn, l.fp, l.messages, k) for l in validation_table.itertuples()]
    return validation_table['threshold'].iloc[int(np.argmin(costs))]   # der Test wird hier nicht angesehen

In [ ]:
# Überprüfung von a) und b), dann das Protokoll
k_med = 2
valid = media_library[media_library['subset'] == 'validation']
test_med = media_library[media_library['subset'] == 'test']
if cost_per_message(3, 18, 150, 2) is None or threshold_choice(valid, k_med) is None:
    print('noch nicht ausgefüllt. Ergänzen Sie die Funktionen.')
else:
    cv = [cost_per_message(l.fn, l.fp, l.messages, k_med) for l in valid.itertuples()]
    print('Validierung, Kosten pro Nachricht:', np.round(cv, 3), '→',
          'richtig' if np.allclose(cv, [39 / 150, 24 / 150, 20 / 150, 21 / 150]) else 'bitte prüfen')
    s = threshold_choice(valid, k_med)
    print(f'auf den Validierungsdaten gewählter Schwellenwert: {s} →', 'richtig' if s == 0.6 else 'bitte prüfen')
    l = test_med[test_med['threshold'] == s].iloc[0]                    # NUR EINE Zeile des Tests
    ct = cost_per_message(l.fn, l.fp, l.messages, k_med)
    print(f'Test, nur beim Schwellenwert {s}: {ct:.3f} pro Nachricht →', 'richtig' if abs(ct - 35 / 240) < 1e-9 else 'bitte prüfen')
    # e) Was der Kollege tut: Er schaut sich ALLE Zeilen des Tests an und behält die beste
    ct_all = [cost_per_message(r.fn, r.fp, r.messages, k_med) for r in test_med.itertuples()]
    s_coll = test_med['threshold'].iloc[int(np.argmin(ct_all))]
    print(f'Angabe des Kollegen: Schwellenwert {s_coll}, {min(ct_all):.3f} pro Nachricht (auf dem Test gewählt: zu optimistisch);',
          f'ehrliche Angabe: Schwellenwert {s}, {ct:.3f} pro Nachricht →',
          'richtig' if s_coll == 0.7 and abs(min(ct_all) - 28 / 240) < 1e-9 else 'bitte prüfen')
    plt.figure(figsize=(5.5, 3))
    plt.plot(valid['threshold'], cv, 'o-', color=SET_VAL, label='Validierung (dient zur Wahl)')
    plt.plot(test_med['threshold'], ct_all, 's--', color=SET_TEST, alpha=0.5, label='Test (bei der Wahl nicht ansehen)')
    plt.scatter([s], [ct], s=120, color=SET_TEST, zorder=3, label='Test, beim gewählten Schwellenwert')
    plt.xlabel('Schwellenwert'); plt.ylabel('Kosten pro Nachricht (k = 2)'); plt.legend(fontsize=8); plt.grid(alpha=0.4); plt.show()

### f) Kommentierte Lösung

Die Mediathek kann angeben: „Beim Schwellenwert 0,6, gewählt auf den Validierungsdaten, kostet der Filter etwa 0,146 pro Nachricht auf Testdaten mit 240 Nachrichten, die für nichts anderes verwendet worden waren.“ Die Grafik zeigt, warum die Angabe des Kollegen zu schön ist: Indem er den tiefsten Punkt der Testkurve wählt, macht er die Testdaten zu Validierungsdaten. Seine 0,117 sind keine Messung der künftigen Kosten mehr. Schließlich liefern 240 Nachrichten keine exakten Kosten: Lektion 6.6 zeigt, wie man angibt, um wie viel sie schwanken können.

## Aufgabe 6.3 — Interpretieren: ein Leck zwischen den Folds

*Begriffe: Datenleck (Lektion 6.2), gruppenweise Kreuzvalidierung und Standardisierung in jeder Runde (Lektion 6.4).* Daten D6.E3: wo die Nachrichten von vier Gruppen in den Folds des Ehrenamtlichen gelandet sind.

### a) bis d) Kommentierte Lösung

**a)** In Runde 1 bilden die Folds 2 und 3 die Trainingsdaten: Der Filter hat $8 + 9 = 17$ Kopien der Kampagne « Montres » (Uhren) gesehen. Er findet 9 davon in Fold 1 wieder, in der Validierung.

**b)** **Schritt 2**: Der Fold wird Nachricht für Nachricht ausgelost. Jede Kampagne wird also auf die Folds zerschnitten (die vier Gruppen der Tabelle sind es). In jeder Runde hat der Filter im Training Kopien gesehen, die den Validierungsnachrichten fast gleichen: Er erkennt die Kampagne wieder, statt eine neue Nachricht zu beurteilen. **Schritt 1**: Mittelwert und Standardabweichung werden auf den 800 Nachrichten berechnet, also einschließlich der Nachrichten jedes Validierungs-Folds. Diese beiden Zahlen tragen ein wenig Information aus der Validierung ins Training. Dieses zweite Leck ist kleiner, aber von derselben Art.

**c)** Der Testverlust, 0,34, beschreibt die künftigen Nachrichten am besten: Die Testdaten wurden nach Kampagnen gebildet und nur einmal verwendet. Der Kreuzvalidierungswert, 0,11, ist zu schön, weil die Validierung bereits gesehene Kopien enthielt. Die Schritte 4 und 5 sind für sich genommen korrekt: $\lambda$ per Kreuzvalidierung wählen, dann die Testdaten ein einziges Mal öffnen. Aber Schritt 4 arbeitet mit verfälschten Werten: Das Leck belohnt Filter, die auswendig lernen, sodass das gewählte $\lambda$ selbst zu klein sein kann.

**d)** Die Gruppen umfassen 26, 15, 22 und 12 Nachrichten, also 75; ideal wären 25 pro Fold. Eine mögliche Verteilung: Fold 1 = Montres (26), Fold 2 = Loterie (22), Fold 3 = Colis en attente und Lettre de la mairie (27). Schritt 1 neu formuliert: „In jeder Runde Mittelwert und Standardabweichung jedes Merkmals nur auf den beiden Trainings-Folds berechnen und dann mit diesen beiden Zahlen die Trainingsnachrichten und die des Validierungs-Folds standardisieren.“

In [ ]:
def split_groups(allocation):
    """allocation: eine Tabelle mit einer Spalte 'group' und den Spalten 'fold_1', 'fold_2', 'fold_3'
    (Anzahl der Nachrichten der Gruppe in jedem Fold). Gibt die Liste der zerschnittenen Gruppen zurück."""
    n_folds = (allocation[['fold_1', 'fold_2', 'fold_3']] > 0).sum(axis=1)   # in wie vielen Folds ist die Gruppe?
    return list(allocation.loc[n_folds > 1, 'group'])

In [ ]:
# Überprüfung von b) und d)
proposal = pd.DataFrame([('Montres', 26, 0, 0), ('Colis en attente', 0, 0, 15),
                            ('Loterie', 0, 22, 0), ('Lettre de la mairie', 0, 0, 12)],
                           columns=['group', 'fold_1', 'fold_2', 'fold_3'])
if split_groups(folds_of_groups) is None:
    print('noch nicht ausgefüllt. Ergänzen Sie die Funktion.')
else:
    print('Folds des Ehrenamtlichen, zerschnittene Gruppen:', split_groups(folds_of_groups), '→',
          'richtig' if len(split_groups(folds_of_groups)) == 4 else 'bitte prüfen')
    print('korrigierte Verteilung, zerschnittene Gruppen:', split_groups(proposal), '| Größen der Folds:',
          proposal[['fold_1', 'fold_2', 'fold_3']].sum().tolist(), '→',
          'richtig' if split_groups(proposal) == [] else 'bitte prüfen')
    print('Runde 1: im Training gesehene Kopien von « Montres » =', int(folds_of_groups.loc[0, ['fold_2', 'fold_3']].sum()),
          ', in der Validierung =', int(folds_of_groups.loc[0, 'fold_1']))

## Aufgabe 6.4 — Kreuzvalidierung nach Kampagnen

*Begriffe: Kreuzvalidierung (Lektion 6.4, Gleichung 6.4), L2-Strafterm (Gleichung 6.2), Standardisierung (Gleichung 2.7), Datenleck (Lektion 6.2).* Die Nachrichten werden von `make_messages()` erzeugt (Import-Zelle). Die Gruppen 30 bis 39 bilden die Testdaten: Sie werden erst in Frage e) angefasst.

In [ ]:
# Die Nachrichten der Aufgabe: 30 Gruppen für die Kreuzvalidierung, 10 Gruppen für den Test zurückgelegt
msgs = make_messages()
test_ = msgs['group'] >= 30
X_cv, y_cv, g_cv = msgs.loc[~test_, FEATURE_NAMES].to_numpy(float), msgs.loc[~test_, 'spam'].to_numpy(float), msgs.loc[~test_, 'group'].to_numpy()
X_test, y_test = msgs.loc[test_, FEATURE_NAMES].to_numpy(float), msgs.loc[test_, 'spam'].to_numpy(float)
print(f'{len(msgs)} Nachrichten, {msgs.group.nunique()} Gruppen; Kreuzvalidierung: {len(y_cv)} Nachrichten; Test: {len(y_test)} Nachrichten')
print(msgs[FEATURE_NAMES].describe().loc[['mean', 'std']].round(1))     # sehr unterschiedliche Einheiten

### a) Kommentierte Lösung

Man erwartet, dass er **erst sinkt und dann wieder steigt**. Ohne Strafterm kann der Filter den sechs Merkmalen ohne Bezug zum Label große Gewichte geben, die zufällig von Gruppe zu Gruppe schwanken: Er passt sich zu stark an. Bei einem zu starken Strafterm werden alle Gewichte plattgedrückt.

In [ ]:
def group_folds(groups, K):
    """Fold-Nummer (0 bis K - 1) jeder Nachricht: Die Gruppen werden, aufsteigend sortiert, reihum
    auf die Folds verteilt; alle Nachrichten einer Gruppe erhalten denselben Fold."""
    ranks = {g: i % K for i, g in enumerate(np.unique(groups))}
    return np.array([ranks[g] for g in groups])


def standardization(X_train, X_val):
    """Standardisiert X_train und X_val mit Mittelwert und Standardabweichung NUR von X_train."""
    m, s = X_train.mean(axis=0), X_train.std(axis=0)
    return (X_train - m) / s, (X_val - m) / s


def cv_score(lam, folds_, K=3):
    """Gleichung 6.4: Mittelwert über die K Runden des Verlusts OHNE Strafterm auf dem Validierungs-Fold."""
    losses = []
    for k in range(K):
        train, val = folds_ != k, folds_ == k
        Xe, Xv = standardization(X_cv[train], X_cv[val])          # Zahlen innerhalb der Runde berechnet, auf den Trainingsdaten
        w, b = fit(Xe, y_cv[train], lam)
        losses.append(mean_loss(y_cv[val], Xv, w, b))     # man beurteilt die Vorhersagen, ohne Strafterm
    return float(np.mean(losses))

In [ ]:
# Überprüfung von b) und c)
LAMBDAS = [0, 0.001, 0.01, 0.1, 1]
p_g = group_folds(g_cv, 3)
if p_g is None or standardization(X_cv, X_cv)[0] is None or cv_score(0.1, p_g) is None:
    print('noch nicht ausgefüllt. Ergänzen Sie die drei Funktionen.')
else:
    splits = sum(len(set(p_g[g_cv == g])) > 1 for g in np.unique(g_cv))
    print(f'Folds nach Gruppen: {np.bincount(p_g).tolist()} Nachrichten, {splits} zerschnittene Gruppe(n) →',
          'richtig' if splits == 0 and set(p_g) == {0, 1, 2} else 'bitte prüfen')
    Xe, Xv = standardization(X_cv[p_g != 0], X_cv[p_g == 0])
    print('standardisiertes Training: Mittelwerte ≈ 0 und Standardabweichungen ≈ 1; Validierung: nicht genau →',
          'richtig' if np.allclose(Xe.mean(0), 0) and np.allclose(Xe.std(0), 1) and not np.allclose(Xv.mean(0), 0) else 'bitte prüfen')
    cv_group = {lam: cv_score(lam, p_g) for lam in LAMBDAS}
    lam_cv = min(cv_group, key=cv_group.get)
    print('KV nach Gruppen:', {l: round(v, 3) for l, v in cv_group.items()}, '→ gewähltes λ:', lam_cv, '→',
          'richtig' if lam_cv == 0.01 else 'bitte prüfen')

In [ ]:
# d) Folds Nachricht für Nachricht ausgelost (nicht nachmachen): das Datenleck aus Lektion 6.2, dreifach
if p_g is not None and cv_score(0.1, p_g) is not None:
    p_m = np.random.default_rng(1).integers(0, 3, len(y_cv))
    cv_message = {lam: cv_score(lam, p_m) for lam in LAMBDAS}
    print('KV Nachricht für Nachricht:', {l: round(v, 3) for l, v in cv_message.items()})
    plt.plot(range(5), list(cv_group.values()), 'o-', color=SET_VAL, label='Folds nach Gruppen')
    plt.plot(range(5), list(cv_message.values()), 's--', color='grey', label='Folds Nachricht für Nachricht (Datenleck)')
    plt.xticks(range(5), [str(l) for l in LAMBDAS]); plt.xlabel('λ'); plt.ylabel('Kreuzvalidierungswert')
    plt.legend(); plt.grid(alpha=0.4); plt.show()

    # e) Endmodell: Standardisierung und Gewichte auf den 30 Gruppen, mit dem gewählten λ; Test NUR EINMAL
    Xc, Xt = standardization(X_cv, X_test)
    w_f, b_f = fit(Xc, y_cv, lam_cv)
    print(f'Testverlust des Endmodells (λ = {lam_cv}): {mean_loss(y_test, Xt, w_f, b_f):.3f}; '
          f'KV nach Gruppen: {cv_group[lam_cv]:.3f}; KV Nachricht für Nachricht: {cv_message[lam_cv]:.3f}')

### d) und e) Kommentierte Lösung

**c)** Mit Folds nach Gruppen beträgt der Kreuzvalidierungswert etwa 0,547 für $\lambda = 0$, 0,493 für 0,001, **0,378 für 0,01**, 0,424 für 0,1 und 0,609 für 1. Er sinkt und steigt dann wieder: Wir behalten $\lambda = 0{,}01$. Ohne Strafterm passt sich der Filter zu stark an die sechs Merkmale ohne Bezug zum Label an; bei $\lambda = 1$ werden die Gewichte zu stark plattgedrückt.

**d)** Mit Folds, die Nachricht für Nachricht ausgelost werden, fallen die Werte auf 0,207, 0,206, 0,246, 0,385 und 0,598. Jede Gruppe wird auf die Folds zerschnitten: In jeder Runde enthält die Validierung fast identische Kopien von Trainingsnachrichten, und der Filter erkennt sie wieder. Der Wert ist zu schön, vor allem für kleine $\lambda$, die das Auswendiglernen belohnen: Diese Aufteilung hätte sogar $\lambda = 0{,}001$ gewählt. Das Leck verfälscht zugleich die Messung und die Wahl.

**e)** Das Endmodell ($\lambda = 0{,}01$, Standardisierung und Gewichte auf den 30 Gruppen berechnet) hat einen Testverlust von 0,356, ein einziges Mal gemessen auf 10 nie gesehenen Gruppen. Er liegt nahe am gruppenweisen Kreuzvalidierungswert 0,378 und weit von den 0,246 der Folds Nachricht für Nachricht. Die gruppenweise Kreuzvalidierung hat ehrlich angekündigt, was den Filter erwartete; die andere lag um ein Drittel daneben. Der verbleibende Unterschied zwischen 0,356 und 0,378 ist Zufall: 94 Testnachrichten sind wenig.

## Aufgabe 6.5 — Vorhersagen, dann überprüfen: das Erinnerungsschreiben des Theaters

*Begriffe: Differenz in Punkten (Gleichung 6.5), Standardfehler (Gleichung 6.6), Konfidenzintervall (Gleichung 6.7).* Daten D6.E5. Das Video lohnt sich nur, wenn es mindestens 0,5 Punkte bringt.

### a) bis e) Kommentierte Lösung

**a)** Rate A: $126 / 1\,800 = 7{,}0\ \%$. Rate B: $147 / 1\,750 = 8{,}4\ \%$. Differenz: $8{,}4 - 7{,}0 = 1{,}4$ Prozentpunkte. Relative Veränderung: $1{,}4 / 7{,}0 = 20\ \%$.

**b)** In Punkten: $7 \times 93 / 1\,800 \approx 0{,}362$ und $8{,}4 \times 91{,}6 / 1\,750 \approx 0{,}440$. $\text{ES} = \sqrt{0{,}362 + 0{,}440} = \sqrt{0{,}801} \approx 0{,}90$ Punkt (ES: Standardfehler). Intervall: $1{,}4 \pm 1{,}96 \times 0{,}90 \approx 1{,}4 \pm 1{,}75$, also etwa von **−0,35 bis 3,15 Punkten**.

**c)** *Das Vorzeichen.* Die gemessene Differenz ist positiv, aber die Null liegt im Intervall, ebenso wie kleine negative Effekte: Der Versuch erlaubt nicht die Aussage, dass das Video zu mehr Reservierungen geführt hat. *Die Genauigkeit.* Das Intervall ist 3,5 Punkte breit, zweieinhalbmal so groß wie die Differenz selbst: Der Effekt ist sehr schlecht bekannt. *Der praktische Nutzen.* Die Schwelle von 0,5 Punkten liegt im Intervall, ebenso wie die Null und Effekte von 3 Punkten: Das Video kann sehr nützlich oder nutzlos sein; die Daten entscheiden es nicht. Der Kollege verwechselt „das Intervall enthält die Null“ mit „der Effekt ist null“: Ein zu kleiner Versuch beweist nicht, dass es keinen Effekt gibt. 1,4 Punkte bleiben der plausibelste Wert; was fehlt, ist Genauigkeit.

**d)** Der Standardfehler verhält sich wie $1 / \sqrt{n}$: Viermal so viele Zuschauer teilen den Standardfehler, und damit die Breite des Intervalls, durch $\sqrt{4} = 2$, nicht durch 4.

**e)** $\text{ES} = \sqrt{0{,}801 / 4} \approx 0{,}45$ Punkt, und das Intervall reicht von $1{,}4 - 0{,}88 \approx 0{,}52$ bis $1{,}4 + 0{,}88 \approx 2{,}28$ Punkten. Die Null wäre ausgeschlossen, die Schwelle von 0,5 Punkten gerade so: Blieben die Raten gleich, würde das Video als nützlich beurteilt, knapp.

In [ ]:
def interval(conv_A, n_A, conv_B, n_B):
    """Gibt (Differenz, untere Grenze, obere Grenze) in Prozentpunkten zurück."""
    pA, pB = conv_A / n_A, conv_B / n_B
    se = np.sqrt(pA * (1 - pA) / n_A + pB * (1 - pB) / n_B)   # Gleichung 6.6
    return 100 * (pB - pA), 100 * (pB - pA - 1.96 * se), 100 * (pB - pA + 1.96 * se)

In [ ]:
# Überprüfung von a), b) und e)
A, B = theater.iloc[0], theater.iloc[1]
r = interval(A['bookings'], A['audience'], B['bookings'], B['audience'])
if r is None:
    print('noch nicht ausgefüllt. Ergänzen Sie die Funktion.')
else:
    print(f'tatsächlicher Versuch: Differenz {r[0]:.2f} Punkte, Intervall von {r[1]:.2f} bis {r[2]:.2f} →',
          'richtig' if abs(r[0] - 1.4) < 1e-9 and abs(r[1] + 0.35) < 0.01 and abs(r[2] - 3.15) < 0.01 else 'bitte prüfen')
    r4 = interval(4 * A['bookings'], 4 * A['audience'], 4 * B['bookings'], 4 * B['audience'])
    print(f'viermal größerer Versuch: von {r4[1]:.2f} bis {r4[2]:.2f}; Breite {r[2] - r[1]:.2f} → {r4[2] - r4[1]:.2f} →',
          'richtig' if abs((r[2] - r[1]) / (r4[2] - r4[1]) - 2) < 1e-9 and r4[1] > 0.5 else 'bitte prüfen')

In [ ]:
# f) 1000 simulierte Versuche mit einem wahren Effekt von 1,4 Punkten: Wie viele Intervalle enthalten 1,4?
rng = np.random.default_rng(65)
nA, nB, true_A, true_B = 1800, 1750, 0.07, 0.084
a = rng.binomial(nA, true_A, 1000) / nA
b = rng.binomial(nB, true_B, 1000) / nB
se_sim = np.sqrt(a * (1 - a) / nA + b * (1 - b) / nB)
contains = np.abs((b - a) - 0.014) <= 1.96 * se_sim
print(f'Anteil der Intervalle, die den wahren Effekt enthalten: {contains.mean():.1%}')
print(f'Anteil der Intervalle, die die Null enthalten: {np.mean(np.abs(b - a) <= 1.96 * se_sim):.1%}')

### f) Kommentierte Lösung

Etwa 94 % der 1000 simulierten Intervalle enthalten den wahren Effekt von 1,4 Punkten: Das sind die „95 %“, bis auf die Zufälligkeit der Simulation. Sie beschreiben die **Methode**: Über sehr viele Versuche hinweg enthalten etwa 95 % der so konstruierten Intervalle den wahren Effekt. Beachten Sie auch, dass bei einem wahren Effekt von 1,4 Punkten ein guter Teil der Intervalle die Null enthält: Ein Versuch dieser Größe übersieht oft einen echten Effekt. Ein Grund mehr, „enthält die Null“ nicht als „kein Effekt“ zu lesen.

## Aufgabe 6.6 — Herausforderung: Wie viele Kunden, und wann hinschauen?

*Begriffe: Standardfehler und Konfidenzintervall (Lektion 6.6), Binomialverteilung (Lektion 4.5).*

### a) bis c) Kommentierte Lösung

**a)** **Ziel der Dimensionierung: die Genauigkeit.** Das 95 %-Intervall soll sich um nicht mehr als $m = 1$ Punkt von der gemessenen Differenz entfernen. Das ist keine Power-Berechnung: Wir legen weder eine zu testende Hypothese noch eine Wahrscheinlichkeit fest, einen Effekt zu entdecken. **Annahmen:** zwei zufällig zugeteilte Gruppen gleicher Größe $n$; angenommene Raten $p_A = 8$ % und $p_B = 10$ %, die aus dem Versuch in Lektion 6.5; Näherung durch die Glockenkurve ($z = 1{,}96$); Intervall ein einziges Mal abgelesen, am Ende.

In Punkten gilt $\text{ES} = \sqrt{\big(p_A(100 - p_A) + p_B(100 - p_B)\big) / n}$, und die Bedingung $z \times \text{ES} \le m$ ergibt:

$$n \ge \frac{z^2 \, \big(p_A(100 - p_A) + p_B(100 - p_B)\big)}{m^2} = \frac{1{,}96^2 \times (8 \times 92 + 10 \times 90)}{1^2} = 3{,}8416 \times 1\,636 \approx 6\,284{,}9 .$$

Man rundet auf die nächste ganze Zahl auf: **6285 Kunden pro Gruppe**, etwa 12 570 insgesamt. Diese 6285 Kunden entsprechen einer **ungefähren** Fehlermarge von einem Punkt: ungefähr, weil die Glockenkurve eine Näherung ist und weil die Rechnung Raten von 8 % und 10 % annimmt. Weichen die wahren Raten ab, ändert sich die Größe: Bei Raten näher an 50 % wären mehr Kunden nötig.

**b)** Das Intervall würde von $2{,}0 - 1{,}0 = 1{,}0$ bis $3{,}0$ Punkten reichen: Es würde die Schwelle berühren, ohne sie auszuschließen. Für eine halbe Breite von 0,5 Punkten muss der Standardfehler halbiert werden, also $n$ vervierfacht: etwa 25 140 Kunden pro Gruppe, also 50 280 insgesamt. Das sind mehr als die 40 000 Abonnenten: Ein einziges Schreiben würde nicht reichen; man müsste den Versuch zum Beispiel über zwei Aussendungen wiederholen.

**c)** Nein: Man erwartet viel mehr als 5 %, denn jeder Abend gibt dem Zufall eine neue Gelegenheit, die Grenze zu überschreiten.

In [ ]:
def group_size(pA, pB, half_width):
    """pA, pB in Prozent; gibt das kleinste ganze n zurück, für das 1,96 × SF ≤ half_width gilt."""
    return int(np.ceil((1.96 / half_width) ** 2 * (pA * (100 - pA) + pB * (100 - pB))))

In [ ]:
# Überprüfung von a) und b)
n1 = group_size(8, 10, 1)
if n1 is None:
    print('noch nicht ausgefüllt. Ergänzen Sie die Funktion.')
else:
    print(f'halbe Breite 1 Punkt: {n1} Kunden pro Gruppe →', 'richtig' if n1 == 6285 else 'bitte prüfen')
    print(f'halbe Breite 0,5 Punkt: {group_size(8, 10, 0.5)} Kunden pro Gruppe, '
          f'{2 * group_size(8, 10, 0.5)} insgesamt')

In [ ]:
# d) 2000 Versuche OHNE Effekt. Zeitplan: 250 Kunden pro Tag und pro Gruppe, 20 Ablesungen (eine jeden Abend,
#    auf den kumulierten Kunden: 250, 500… 5000 pro Gruppe). Abbruchregel: erster Abend, an dem das Intervall die 0 ausschließt.
rng = np.random.default_rng(66)
trials, days, per_day, p = 2000, 20, 250, 0.08
cA = rng.binomial(per_day, p, size=(trials, days)).cumsum(axis=1)     # kumulierte Bestellungen, Gruppe A
cB = rng.binomial(per_day, p, size=(trials, days)).cumsum(axis=1)     # Gruppe B: gleiche Rate, kein Effekt
n = per_day * np.arange(1, days + 1)                                  # Kunden pro Gruppe am Ende jedes Tages
pA_, pB_ = cA / n, cB / n
se_j = np.sqrt(pA_ * (1 - pA_) / n + pB_ * (1 - pB_) / n)
excludes = np.abs(pB_ - pA_) > 1.96 * se_j                                # schließt das Intervall des Abends die Null aus?
end, one_evening = excludes[:, -1].mean(), excludes.any(axis=1).mean()
print(f'Intervall nur am Tag 20 abgelesen: {end:.1%} der Versuche schließen fälschlich auf einen Effekt')
print(f'Intervall jeden Abend abgelesen, Abbruch beim ersten « Erfolg »: {one_evening:.1%}')

### d) und e) Kommentierte Lösung

Bedingungen der Simulation: 2000 Versuche ohne Effekt (Rate von 8 % in beiden Gruppen), 250 neue Kunden pro Tag und pro Gruppe, höchstens 20 Ablesungen, eine jeden Abend vom 1. bis zum 20. Tag, auf den seit Beginn kumulierten Kunden (von 250 bis 5000 pro Gruppe); Abbruchregel: Man bricht am ersten Abend ab, an dem das 95 %-Intervall die Null ausschließt, auf der einen oder der anderen Seite, und schließt auf einen Effekt. Nur ein einziges Mal abgelesen, am 20. Abend, schließt das Intervall die Null in etwa 5 % der Versuche ohne Effekt aus (4,6 % in der Simulation): Das ist das vorgesehene Risiko. Mit der Ablesung an jedem Abend und der Abbruchregel schließt man in etwa einem von vier Versuchen fälschlich auf einen Effekt (24,7 %). Diese Zahl hängt vom Zeitplan ab: Mehr Ablesungen oder dichter aufeinanderfolgende Ablesungen würden sie weiter erhöhen. Jeder Abend ist für den Zufall eine neue Gelegenheit, das Intervall von der Null wegzuschieben; in diesem Moment aufzuhören heißt, die günstigen Gelegenheiten zu behalten und die anderen wegzuwerfen. Die Regel: **Größe des Versuchs und Datum der Ablesung im Voraus festlegen**, zum Beispiel 6300 Kunden pro Gruppe nach a), und das Intervall zu diesem Datum ablesen. Zwischendurch auf die Zahlen zu schauen ist nicht verboten, darf aber nicht über den Abbruch entscheiden.